# Seminar 3 — Pandas: grupare, agregare, merge si vizualizare
### Pachete Software — Info, anul III (2026)

---

**Format:** Lucru in echipe de 2-3 persoane 

**Regula de baza:** nu doar *rulati* codul — **explicati** ce face fiecare linie.

## Bloc 0 

### Fisiere de date
Asigurati-va ca urmatoarele fisiere sunt in acelasi director:
- `phone_data.csv` — date despre apeluri, SMS si trafic de date
- `user_usage.csv`, `user_device.csv`, `supported_devices.csv` — date despre utilizatori si dispozitive
- `clienti_leasing20.csv` — clienti leasing (de la S2)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Verificare rapida — fisierele se incarca corect?
df = pd.read_csv("phone_data.csv")
print(f"phone_data.csv: {df.shape[0]} randuri, {df.shape[1]} coloane")
print(f"Coloane: {list(df.columns)}")
print(df.head(3))

---
## Bloc 1 — Conversie date si GroupBy de baza

### Conversia coloanei `date` in format datetime

Pandas stocheaza datele calendaristice ca `object` (string) la import. Pentru a putea face operatii cu ele (filtrare pe intervale, extragere an/luna/zi), trebuie convertite:

In [ ]:
import pandas as pd
import dateutil

df = pd.read_csv("phone_data.csv")
print("Inainte de conversie:")
print(df.dtypes)
print("-" * 40)

# Conversie: string -> datetime (dayfirst=True deoarece formatul este ZZ/LL/AA)
df["date"] = df["date"].apply(dateutil.parser.parse, dayfirst=True)

print("Dupa conversie:")
print(df.dtypes)
print("-" * 40)
print(df[["date"]].head())

### Functia `groupby()` — gruparea inregistrarilor

`groupby()` imparte DataFrame-ul in grupuri pe baza valorilor unei coloane, apoi putem aplica functii pe fiecare grup (sum, mean, count, min, max etc.).

In [ ]:
import pandas as pd
df = pd.read_csv("phone_data.csv")

# Ce grupuri exista in coloana 'item'?
print("Grupuri dupa 'item':")
print(list(df.groupby("item").groups.keys()))

# Cate inregistrari are fiecare grup?
print("\nNumar inregistrari per item:")
print(df.groupby("item").size())

In [ ]:
import pandas as pd
df = pd.read_csv("phone_data.csv")

# Suma duratelor pe luna
print("Durata totala per luna:")
print(df.groupby("month")["duration"].sum())
print("-" * 40)

# Durata totala a apelurilor per retea
print("Durata apeluri per retea:")
apeluri = df[df["item"] == "call"]
print(apeluri.groupby("network")["duration"].sum().sort_values(ascending=False))

In [ ]:
import pandas as pd
df = pd.read_csv("phone_data.csv")

# Grupare dupa mai multe coloane
print("Numar de inregistrari per luna si tip (item):")
print(df.groupby(["month", "item"])["date"].count())

### Challenge 1 — Statistici pe grupuri (5 min)

Folosind `phone_data.csv`:

1. Cate apeluri (`item == "call"`) au fost efectuate in total?
2. Care este durata medie a unui apel? Dar durata maxima?
3. Grupati datele dupa `month` si afisati durata medie per luna
4. Grupati datele dupa `network` si afisati numarul de inregistrari per retea

Aveti **5 minute**.

In [ ]:
# Challenge 1 — Scrieti solutia aici:
import pandas as pd
df = pd.read_csv("phone_data.csv")

total_calls = df[df["item"] == "call"].shape[0]
print(total_calls)

df_calls = df[df["item"] == "call"]
medie = df_calls["duration"].mean()
maxim = df_calls["duration"].max()

print(medie)
print(maxim)

medie_luna = df.groupby("month")["duration"].mean()



---
## Bloc 2 — Agregari complexe cu `agg()` 

Metoda `agg()` permite aplicarea mai multor functii simultan, pe coloane diferite.

In [ ]:
import pandas as pd
df = pd.read_csv("phone_data.csv")

# Agregare: mai multe functii pe coloane diferite
rezultat = df.groupby(["month", "item"]).agg({
    "duration": "sum",           # suma duratelor
    "network_type": "count",     # numarul de inregistrari
    "date": "first"              # prima data din grup
})

print(rezultat)

In [ ]:
import pandas as pd
df = pd.read_csv("phone_data.csv")

# Mai multe functii pe aceeasi coloana
rezultat = df.groupby(["month", "item"]).agg({
    "duration": ["min", "max", "sum", "mean"],
    "network_type": "count",
    "date": ["first", "nunique"]
})

print(rezultat)

In [ ]:
import pandas as pd
df = pd.read_csv("phone_data.csv")

# Salvare rezultat agregate intr-un fisier CSV
rezultat = df.groupby(["month", "item"]).agg({"duration": "sum", "network_type": "count"})
rezultat.to_csv("agregare.csv")
print("Salvat in agregare.csv")
print(pd.read_csv("agregare.csv"))

### Challenge 2 — Raport lunar (5 min)

Folosind `phone_data.csv`, creati un raport care contine, **pentru fiecare luna**:

1. Numarul total de apeluri, SMS-uri si sesiuni de date
2. Durata totala si durata medie a apelurilor
3. Numarul de retele distincte utilizate (`nunique` pe `network`)

Salvati rezultatul intr-un fisier `raport_lunar.csv`.

Aveti **5 minute**.

In [ ]:
# Challenge 2 — Scrieti solutia aici:
import pandas as pd

df = pd.read_csv("phone_data.csv")
df['date'] = pd.to_datetime(df['date'], dayfirst=True)

rezultat_final = df.groupby('month').agg({
    'item' : 'count',
    'duration': ['sum', 'count'],                
    'network': 'nunique'           
})

print(rezultat_final)
rezultat_final.to_csv("raport_lunar.csv")

---
## Bloc 3 — Merge / Join: combinarea DataFrames 

Functia `pd.merge()` combina doua DataFrames pe baza unor coloane comune, similar cu operatia JOIN din SQL.

### Tipuri de merge

| Tip | Descriere | Echivalent SQL |
|-----|-----------|---------------|
| `inner` | Doar randurile cu cheie comuna in ambele tabele | INNER JOIN |
| `left` | Toate randurile din stanga + potrivirile din dreapta | LEFT JOIN |
| `right` | Toate randurile din dreapta + potrivirile din stanga | RIGHT JOIN |
| `outer` | Toate randurile din ambele tabele | FULL OUTER JOIN |

Vom folosi `user_usage.csv` si `user_device.csv` care au coloana comuna `use_id`.

In [ ]:
import pandas as pd

df_usage = pd.read_csv("user_usage.csv")
df_device = pd.read_csv("user_device.csv")

print(f"user_usage:  {df_usage.shape}  | Coloane: {list(df_usage.columns)}")
print(f"user_device: {df_device.shape} | Coloane: {list(df_device.columns)}")
print(f"\nuse_id comune: {df_usage['use_id'].isin(df_device['use_id']).sum()} din {len(df_usage)}")

### Inner merge — doar inregistrarile cu cheie comuna

In [ ]:
import pandas as pd
df_usage = pd.read_csv("user_usage.csv")
df_device = pd.read_csv("user_device.csv")

result = pd.merge(df_usage,
                  df_device[["use_id", "platform", "device"]],
                  on="use_id")  # implicit: how="inner"

print(f"Rezultat inner merge: {result.shape}")
print(result.head())

### Left merge — toate randurile din stanga

In [ ]:
import pandas as pd
df_usage = pd.read_csv("user_usage.csv")
df_device = pd.read_csv("user_device.csv")

result = pd.merge(df_usage,
                  df_device[["use_id", "platform", "device"]],
                  on="use_id",
                  how="left")

print(f"Rezultat left merge: {result.shape}")
print(f"Valori NaN in 'platform': {result['platform'].isnull().sum()}")
print(result.head())

### Outer merge cu indicator

In [ ]:
import pandas as pd
df_usage = pd.read_csv("user_usage.csv")
df_device = pd.read_csv("user_device.csv")

result = pd.merge(df_usage,
                  df_device[["use_id", "platform", "device"]],
                  on="use_id",
                  how="outer",
                  indicator=True)  # adauga coloana _merge

print(f"Rezultat outer merge: {result.shape}")
print(f"\nDistributie sursa:")
print(result["_merge"].value_counts())

### Challenge 3 — Alaturare date (7 min)

1. Faceti un **left merge** intre `user_usage` si `user_device` pe `use_id`
2. Afisati cati utilizatori au platforma `android` si cati au `ios`
3. Calculati traficul mediu (`monthly_mb`) per platforma (android vs ios)
4. Afisati utilizatorii care nu au un dispozitiv asociat (unde `platform` este `NaN`)

Aveti **7 minute**.

In [ ]:
# Challenge 3 — Scrieti solutia aici:
import pandas as pd
df_usage = pd.read_csv("user_usage.csv");
df_device = pd.read_csv("user_device.csv");

df = pd.merge(df_usage, df_device[["use_id", "platform", "device"]], on="use_id", how="left")
count_platform = df["platform"].value_counts
print(count_platform)

trafic_mediu = df.groupby("platform")["monthly_mb"].mean()
print(trafic_mediu)

nan = df[df["platform"].isnull()]
print(nan)

---
## Bloc 4 — Merge pe mai multe DataFrames si analiza 

### Merge cu trei seturi de date

Putem inlantui merge-uri pentru a combina mai multe tabele. Coloana `device` din `user_device` corespunde coloanei `Model` din `supported_devices`.

In [ ]:
import pandas as pd

df_usage = pd.read_csv("user_usage.csv")
df_device = pd.read_csv("user_device.csv")
df_supported = pd.read_csv("supported_devices.csv")

# Pas 1: merge usage + device
result = pd.merge(df_usage,
                  df_device[["use_id", "platform", "device"]],
                  on="use_id",
                  how="left")

# Pas 2: merge cu supported_devices (coloane diferite -> left_on / right_on)
df_supported = df_supported.rename(columns={"Retail Branding": "manufacturer"})
result = pd.merge(result,
                  df_supported[["manufacturer", "Model"]],
                  left_on="device",
                  right_on="Model",
                  how="left")

print(f"Rezultat final: {result.shape}")
print(result.head())

### GroupBy + Agg pe setul combinat

In [ ]:
# Statistici per producator (manufacturer)
import pandas as pd

df_usage = pd.read_csv("user_usage.csv")
df_device = pd.read_csv("user_device.csv")
df_supported = pd.read_csv("supported_devices.csv")

result = pd.merge(df_usage,
                  df_device[["use_id", "platform", "device"]],
                  on="use_id", how="left")

df_supported = df_supported.rename(columns={"Retail Branding": "manufacturer"})
result = pd.merge(result,
                  df_supported[["manufacturer", "Model"]],
                  left_on="device", right_on="Model", how="left")

stats = result.groupby("manufacturer").agg({
    "outgoing_mins_per_month": "mean",
    "outgoing_sms_per_month": "mean",
    "monthly_mb": "mean",
    "use_id": "count"
}).rename(columns={"use_id": "numar_utilizatori"})

print(stats.sort_values("numar_utilizatori", ascending=False).head(10))

### Challenge 4 — Analiza multi-dataset 

Pornind de la merge-ul celor 3 DataFrames (usage + device + supported):

1. Afisati top 5 producatori dupa numarul de utilizatori
2. Care producator are cel mai mare trafic mediu lunar (`monthly_mb`)?
3. Afisati minutele medii per platforma (`platform`: android vs ios)
4. Salvati tabelul cu statistici per producator in `stats_producatori.csv`

Aveti **7 minute**.

In [ ]:
# Challenge 4 — Scrieti solutia aici:

import pandas as pd
df_usage = pd.read_csv("user_usage.csv")
df_device = pd.read_csv("user_device.csv")
df_supported = pd.read_csv("supported_devices.csv")

result = pd.merge(df_usage,
                 df_device[['use_id', 'platform', 'device']],
                 on='use_id', how='left')
df_supported = df_supported.rename(columns={'Retail Branding': 'manufacturer'})
result = pd.merge(result,
                 df_supported[['manufacturer', 'Model']],
                 left_on='device',
                 right_on='Model',
                 how='left')

cerinta1 = result["manufacturer"].value_counts().head(5)
print(cerinta1)

cerinta2 = result.groupby("manufacturer")["monthly_mb"].mean().sort_values(ascending=False)
print(cerinta2)

cerinta3 = result.groupby("platform")["use_actual_mins"].mean()
print("\nMinute medii per platformă:\n", cerinta3)

stats_producatori = result.groupby("manufacturer").agg({
    'monthly_mb': 'mean',
    'use_actual_mins': 'mean',
    'use_id': 'count'
}).rename(columns={'use_id': 'user_count'})

stats_producatori.to_csv("stats_producatori.csv")
print("\nFișierul 'stats_producatori.csv' a fost salvat cu succes!")

---
## Bloc 5 — Vizualizare cu matplotlib

`matplotlib.pyplot` este biblioteca standard pentru grafice in Python. Pandas ofera si metoda `.plot()` direct pe DataFrames, care foloseste matplotlib in spate.

### Grafic cu bare (bar chart)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("clienti_leasing20.csv")

df["AGE"].plot(kind="bar", figsize=(10, 4), color="steelblue")
plt.xlabel("Index client")
plt.ylabel("Varsta")
plt.title("Varsta clientilor")
plt.tight_layout()
plt.show()

### Histograma

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("clienti_leasing20.csv")

df["AGE"].plot(kind="hist", bins=8, color="coral", edgecolor="black")
plt.xlabel("Varsta")
plt.title("Distributia varstei clientilor")
plt.tight_layout()
plt.show()

### Grafic cu bare pe date grupate si sortate

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("clienti_leasing20.csv")

# Venitul total per job, doar pentru barbati, sortat
date_grafic = df[df["SEX"] == "m"].groupby("JOB")["INCOME_PER_YEAR"].sum()
date_grafic.sort_values().plot(kind="bar", color="teal")
plt.ylabel("Venit total")
plt.title("Venit total per ocupatie (barbati)")
plt.tight_layout()
plt.show()

### Grafic circular (pie chart)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("phone_data.csv")

# Distributia tipurilor de comunicare (call, sms, data)
distributie = df.groupby("item")["duration"].sum()
distributie.plot(kind="pie", autopct="%1.1f%%", figsize=(6, 6))
plt.title("Distributia duratelor pe tip")
plt.ylabel("")  # ascunde eticheta axei y
plt.tight_layout()
plt.show()

### Challenge 5 — Vizualizare date (7 min)

Creati **doua grafice** folosind `phone_data.csv`:

1. **Grafic cu bare:** durata totala a SMS-urilor (`item == "sms"`) per luna, sortata crescator
2. **Grafic pie:** repartitia numarului de inregistrari per `network_type` (mobile, landline, etc.)

Fiecare grafic trebuie sa aiba titlu si etichete pe axe.

Aveti **7 minute**.

In [ ]:
# Challenge 5 — Scrieti solutia aici:



---
## Bloc 6 — Challenge final (10 min)

### Challenge 6 (Capstone) — Analiza completa (10 min)

Combinati tot ce ati invatat astazi. Folosind `phone_data.csv`:

1. Convertiti coloana `date` in datetime
2. Creati un raport grupat pe **luna si tip** (`month`, `item`) care contine:
   - Durata totala, medie si maxima
   - Numarul de inregistrari
   - Numarul de retele distincte
3. Creati un **grafic cu bare** al duratei totale a apelurilor per luna
4. Creati un **grafic pie** al duratei totale pe tip de retea (`network_type`) doar pentru apeluri
5. Salvati raportul de la punctul 2 in `analiza_completa.csv`

Aveti **10 minute**. Echipa cu analiza cea mai completa prezinta.

In [ ]:
# Challenge 6 (Capstone) — Scrieti solutia aici:



---
## Bloc 7 — Wrap-up (3 min)

### Recapitulare

| Concept | Elemente cheie |
|---------|---------------|
| **Conversie date** | `dateutil.parser.parse()`, `pd.to_datetime()` |
| **GroupBy** | `groupby()`, `size()`, `sum()`, `mean()`, `count()` |
| **Agregare** | `agg()` — functii multiple pe coloane diferite |
| **Merge** | `pd.merge()` — inner, left, right, outer, `indicator` |
| **Merge avansat** | `left_on`/`right_on`, inlantuire merge-uri |
| **Vizualizare** | `matplotlib` — bar, hist, pie, titlu, etichete |

### Teme
- Terminati challenge-urile ramase
- Documentatie Pandas GroupBy: [pandas.pydata.org/docs](https://pandas.pydata.org/docs/user_guide/groupby.html)
- Galerie matplotlib: [matplotlib.org/gallery](https://matplotlib.org/stable/gallery/index.html)

### Referinte
- J. VanderPlas, *Python Data Science Handbook*: [jakevdp.github.io](https://jakevdp.github.io/PythonDataScienceHandbook/)
- Pandas merge tutorial: [shanelynn.ie](https://www.shanelynn.ie/merge-join-dataframes-python-pandas-index-1/)
- Pandas aggregation: [shanelynn.ie](https://www.shanelynn.ie/summarising-aggregation-and-grouping-data-in-python-pandas/)

---
*Seminar Pachete Software — CSIE, Info anul III (2026)*